# Tokenization


## step-1 : Generating tokens

In [5]:
with open("../Data/verdict.txt","r",encoding="utf-8") as f:
    verdict = f.read()
print(f"Total number of characters present in the file:{len(verdict)}")

Total number of characters present in the file:20781


### Regullar expression to sepereate the sencente with with spaces

In [6]:
 
import re 
text = "Hello, World! , this is a sample text."
result=re.split(r"(\s)",text)

print(result)

['Hello,', ' ', 'World!', ' ', ',', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'sample', ' ', 'text.']


### Modified regular expression to seperate symbols from words counted as tokens

In [7]:
 
result=re.split(r'([,.!]|\s)',text)

print(result)

['Hello', ',', '', ' ', 'World', '!', '', ' ', '', ',', '', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'sample', ' ', 'text', '.', '']


### Remove the white spaces from the result 

In [8]:

result=[item for item in result if item.strip()]

print(result)

['Hello', ',', 'World', '!', ',', 'this', 'is', 'a', 'sample', 'text', '.']


In [9]:
text = "Hello, World! . Is this a -- test?"
result=re.split(r"([,.?!:;_()|']|--|\s)",text)
result=[item.strip() for item in result if item.strip()]

print(result)

['Hello', ',', 'World', '!', '.', 'Is', 'this', 'a', '--', 'test', '?']


## Tokenizing the data in the book verdict


In [10]:
preprocessed=re.split(r'([,.?!:;_()"|\']|--|\s)',verdict)
preprocessed=[item.strip() for item in preprocessed if item.strip()]

print(preprocessed[:20])

['THE', 'VERDICT', 'June', '1908', 'I', 'had', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough']


In [11]:
print(len(preprocessed))

4667


## step-2 : Creating Token IDs


### Count the no.of unique tokens in the data

In [12]:
all_words=sorted(set(preprocessed))
vocabulary_size=len(all_words)

print(vocabulary_size)

1148


### Tokenized text 
#### (Each Unique token is mapped to an unique integer called token IDs)

In [13]:
vocabulary={token:integer for integer,token in enumerate(all_words)}

In [14]:
for i,item in enumerate(vocabulary.items()):
    print(item)
    if i>20:
        break 

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
('1908', 8)
(':', 9)
(';', 10)
('?', 11)
('A', 12)
('AM', 13)
('Ah', 14)
('Among', 15)
('And', 16)
('Are', 17)
('Arrt', 18)
('As', 19)
('At', 20)
('Be', 21)


## Adding Special Context Tokens
##### In case if user is given a word which is not present in the vocabulary , it throws the error.
##### To overcome the error will modify the tokenizer to handle unkown words by attend "<|endoftext|>" and "<|unk|>" tokens in vocabulary

In [15]:
all_words=sorted(set(preprocessed))
all_words.extend(["<|endoftext|>","<|unk|>"])
vocabulary_size=len(all_words)
print(vocabulary_size)

1150


In [16]:
vocabulary={token:integer for integer,token in enumerate(all_words)}
# for i,item in enumerate(vocabulary.items()):
#     if i >= vocabulary_size-2:
#         print(item)
for i,item in enumerate(list(vocabulary.items())[-2::]):
        print(item)

('<|endoftext|>', 1148)
('<|unk|>', 1149)


## Byte-Pair Encoding

In [17]:
! pip install tiktoken

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import importlib
import tiktoken
print("tiktoken version:",importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [19]:
tokenizer=tiktoken.get_encoding("gpt2")

In [20]:
text=("Hi this is a radom text. <|endoftext|> this is a random word dksaimanikantakjdafk.")
integers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)

[17250, 428, 318, 257, 2511, 296, 2420, 13, 220, 50256, 428, 318, 257, 4738, 1573, 288, 591, 47840, 1134, 415, 461, 73, 67, 1878, 74, 13]


In [21]:
string=tokenizer.decode(integers)
print(string)

Hi this is a radom text. <|endoftext|> this is a random word dksaimanikantakjdafk.


## Creating input-target Pairs

#### This is the last step before vector embbeding.

In [33]:
import tiktoken

with open("../Data/verdict.txt","r",encoding="utf-8") as f:
    verdict = f.read()

tokenizer = tiktoken.get_encoding("gpt2")

enc_text = tokenizer.encode(verdict)
print(len(enc_text))

5774


In [34]:
enc_sample=enc_text[50:]

In [41]:
context_size=5
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"x : {x}")
print(f"y : {y}")

x : [11, 339, 550, 5710, 465]
y : [339, 550, 5710, 465, 12036]


In [42]:
for i in range(1,context_size+1):
    x=enc_sample[:i]
    y=enc_sample[i]
    print(f"Input:{x} ---> Target:{y}")

Input:[11] ---> Target:339
Input:[11, 339] ---> Target:550
Input:[11, 339, 550] ---> Target:5710
Input:[11, 339, 550, 5710] ---> Target:465
Input:[11, 339, 550, 5710, 465] ---> Target:12036


In [43]:
for i in range(1,context_size+1):
    x=enc_sample[:i]
    y=enc_sample[i]
    print(f"Input: {tokenizer.decode(x)} ---> Target:{tokenizer.decode([y])}")

Input: , ---> Target: he
Input: , he ---> Target: had
Input: , he had ---> Target: dropped
Input: , he had dropped ---> Target: his
Input: , he had dropped his ---> Target: painting
